# Stage 0 - inspect (Kaggle, GPU T4 x2 or CPU, Internet ON)
Paste the full output of the last two cells back to the assistant. Expected time: 5-15 min (mostly the 2B SAE download, ~6.5 GB). **Untested.**

In [1]:
import os, subprocess, sys
REPO = "https://github.com/sarrazer24/reins-mini"
if not os.path.isdir("/kaggle/working/reins-mini"):
    !git clone {REPO} /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!pip install -q -U transformers accelerate huggingface_hub
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS      # authors' data + frozen controllers; deleted at the end
GUISE = "/tmp/REINS/guise"; CTRL = "/tmp/REINS/configs/qwen35_2b_guise.controllers.json"
from huggingface_hub import snapshot_download
SAE_ROOT = snapshot_download("Carlos4869/REINS-SAE", allow_patterns=["qwen3.5-2b-base/*"], local_dir="/tmp/sae") + "/qwen3.5-2b-base"
print(os.listdir(SAE_ROOT)[:5])

Cloning into '/kaggle/working/reins-mini'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 43 (delta 0), reused 43 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 49.26 KiB | 1.97 MiB/s, done.
/kaggle/working/reins-mini
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

['layer_02.pt', 'layer_11.pt', 'layer_23.pt', 'layer_16.pt', 'layer_10.pt']


In [2]:
!python scripts/01_check_paper_tables.py | tail -3
!python scripts/02_inspect_guise.py --guise-dir {GUISE} --authors-clone /tmp/REINS

[PASS] Table10 rowsum REINS-Gate: sum=100.00

42/42 checks passed
samples: 900 | unique ids: 900
categories: {'hate': 180, 'cybercrime': 180, 'violence': 180, 'pornography': 180, 'autolesion': 180}
missing matched-safe prompt: 0
my split seed=12: calibration=600 evaluation=300 {'autolesion': 60, 'cybercrime': 60, 'hate': 60, 'pornography': 60, 'violence': 60}
wrote results/eval_split_seed12.json (ids only, no text)
evaluation ids identical to authors' split function: True


In [3]:
!python scripts/03_inspect_sae_ckpt.py --sae-root {SAE_ROOT} --layers 0,14,23 --d-model 2048 --width 16384

files: ['layer_00.pt', 'layer_01.pt', 'layer_02.pt', 'layer_03.pt', 'layer_04.pt', 'layer_05.pt', 'layer_06.pt', 'layer_07.pt', 'layer_08.pt', 'layer_09.pt', 'layer_10.pt', 'layer_11.pt', 'layer_12.pt', 'layer_13.pt', 'layer_14.pt', 'layer_15.pt', 'layer_16.pt', 'layer_17.pt', 'layer_18.pt', 'layer_19.pt', 'layer_20.pt', 'layer_21.pt', 'layer_22.pt', 'layer_23.pt']

== layer 0: /tmp/sae/qwen3.5-2b-base/layer_00.pt (269 MB)
dict
   b_dec tensor(2048,) torch.float32
   k tensor() torch.int32
   threshold tensor() torch.float32
   decoder.weight tensor(2048, 16384) torch.float32
   encoder.weight tensor(16384, 2048) torch.float32
   encoder.bias tensor(16384,) torch.float32
loader OK: n,d = 16384 2048 threshold = 0.2872622311115265 input_scale = None | L0 on random input: 6229.75

== layer 14: /tmp/sae/qwen3.5-2b-base/layer_14.pt (269 MB)
dict
   b_dec tensor(2048,) torch.float32
   k tensor() torch.int32
   threshold tensor() torch.float32
   decoder.weight tensor(2048, 16384) torch.floa

In [4]:
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
print("transformers", transformers.__version__, "| gpus", torch.cuda.device_count())
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-2B-Base")
m = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-2B-Base", torch_dtype=torch.float16).cuda().eval()
import sys; sys.path.insert(0, "src")
from reinsmini import steer as S
print(S.find_layer_list(m, 24)[0], "| decoder blocks: 24")
ids = tok("Question: What is the capital of France?\n\nAnswer:", return_tensors="pt").to("cuda")
print(tok.decode(m.generate(**ids, max_new_tokens=30, do_sample=False)[0]))

transformers 5.23.0 | gpus 2


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

model.layers | decoder blocks: 24


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Question: What is the capital of France?

Answer: Paris

Explanation: Paris is the capital city of France. It is located in the Île-de-France region, which is the most populous region


In [5]:
!rm -rf /tmp/REINS /tmp/sae